### data ingestion to vector db pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### read all pdfs inside the directory
def process_all_pds(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f"error: {e}")

    print(f"\ntotal documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pds("../data")


C:\Users\adity\AppData\Local\Temp\ipykernel_4168\838160545.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\RECEPTION\developments\repo\traditional-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


found 1 PDF files to process

Processing: System Design Interview by Alex Xu.pdf
loaded 269 pages

total documents loaded:269


In [2]:
### chunking (text spliting)
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better rag performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents=documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunks=split_documents(documents=all_pdf_documents)

split 269 documents into 440 chunks

Example chunk:
Content: System Design Interview: An Insider’s Guide
All rights reserved. This book or any portion thereof may not be reproduced or used in any
manner whatsoever without the express written permission of the p
Metadata: {'producer': 'GPL Ghostscript 9.27', 'creator': 'calibre 3.9.0 [https://calibre-ebook.com]', 'creationdate': '2020-12-19T00:59:24-05:00', 'moddate': '2026-04-21T23:08:31+05:30', 'author': 'Alex Xu', 'title': "System Design Interview – An insider's guide, Second Edition: Step by Step Guide, Tips and 15 System Design Interview Questions with Detailed Solutions", 'source': '..\\data\\pdf_files\\System Design Interview by Alex Xu.pdf', 'total_pages': 269, 'page': 1, 'page_label': '2', 'source_file': 'System Design Interview by Alex Xu.pdf', 'file_type': 'pdf'}


### embedding and vector store db

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name:str='all-MiniLM-L6-v2'):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded. embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts:List[str])-> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: lists of text string to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("model not loaded")

        print(f"generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        embeddings=np.asarray(embeddings)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings


### vectorstore

In [4]:
from typing import Optional
from chromadb.api import ClientAPI
from chromadb.api.models.Collection import Collection
import hashlib
import os

class VectorStore:
    """Manages document embeddings in a ChromaDB vectorstore"""
    def __init__(self, collection_name:str="pdf_documents", persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: directory to persist the vector store 
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client: Optional[ClientAPI]=None
        self.collection: Optional[Collection]=None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromadb client and collection"""
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
    
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"vector store initialized. collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error initializing vector store: {e}")
            raise

    def reset(self):
        """Delete the collection and recreate it empty (clears duplicate data)"""
        if self.client is None:
            raise ValueError("client not initialized")
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            pass  # collection may not exist yet
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"PDF document embeddings for RAG"}
        )
        print(f"collection '{self.collection_name}' reset. documents: {self.collection.count()}")

    def add_documents(self, documents:List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings into the vector store

        Uses a content-based id and upsert, so re-running this method overwrites
        existing chunks instead of inserting duplicates.

        Args:
            documents: list of langchain documents
            embeddings: corresponding embeddings for the documents
        """
        if self.collection is None:
            raise ValueError("collection not initialized")

        if len(documents)!= len(embeddings):
            raise ValueError("number of documents must match number of embeddings")

        print(f"adding {len(documents)} documents to vectorstores")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # deterministic id from content + source + page -> idempotent ingestion
            fingerprint=(
                str(doc.metadata.get("source",""))
                + str(doc.metadata.get("page",""))
                + doc.page_content
            )
            doc_id="doc_"+hashlib.sha1(fingerprint.encode("utf-8")).hexdigest()[:16]
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successfully upserted {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error adding documents to vector store: {e}")

In [5]:
vectorstore=VectorStore()
embedding_manager=EmbeddingManager()

### clear any previously ingested (and possibly duplicated) data, then ingest once
vectorstore.reset()

### convert text to embeddings
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

vector store initialized. collection: pdf_documents
existing documents in collection: 1760
loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16698.75it/s]


model loaded. embedding dimension: 384
collection 'pdf_documents' reset. documents: 0
generating embeddings for 440 texts...


Batches: 100%|██████████| 14/14 [00:04<00:00,  3.21it/s]


generated embeddings with shape: (440, 384)
adding 440 documents to vectorstores
successfully upserted 440 documents to vector store
total documents in collection: 440


In [6]:
### rag retriever
class RAGRetriever:
    """Handled query-based retrieval from the vector store"""

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self, query:str, top_k:int=5, score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: the search query
            top_k: no. of top results to return
            score_threshold:minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        collection=self.vector_store.collection
        if collection is None:
            raise ValueError("collection not initialized")
        try:
            results=collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]
            doc_lst=results['documents']
            meta_lst=results['metadatas']
            dist_lst=results['distances']
            id_lst=results['ids']
            if doc_lst and meta_lst and dist_lst and id_lst:
                documents=doc_lst[0]
                metadatas=meta_lst[0]
                distances=dist_lst[0]
                ids=id_lst[0]

                for i, (doc_id, doc, meta, dist) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score=1-dist
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':doc,
                            'metadata':meta,
                            'similarity_score':similarity_score,
                            'distance':dist,
                            'rank':i+1
                        })

                print(f"retrieved {len(retrieved_docs)} documents (after filter)")
            else:
                print("no docs found")

            return retrieved_docs
            
        except Exception as e:
            print(f"error: {e}")
            raise
        

In [7]:
rag_retriever=RAGRetriever(vector_store=vectorstore,embedding_manager=embedding_manager)
rag_retriever.retrieve(query="what are different techniques of rate-limiting?")

Retrieving documents for query: what are different techniques of rate-limiting?
Top K: 5, Score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 124.67it/s]

generated embeddings with shape: (1, 384)
retrieved 5 documents (after filter)


[{'id': 'doc_99213c64c1df403a',
  'content': 'CHAPTER 4: DESIGN A RATE LIMITER \nIn a network system, a rate limiter is used to control the rate of traffic sent by a client or a\nservice. In the HTTP world, a rate limiter limits the number of client requests allowed to be\nsent over a specified period. If the API request count exceeds the threshold defined by the\nrate limiter, all the excess calls are blocked. Here are a few examples:\n• A user can write no more than 2 posts per second.\n• You can create a maximum of 10 accounts per day from the same IP address.\n• You can claim rewards no more than 5 times per week from the same device.\nIn this chapter, you are asked to design a rate limiter. Before starting the design, we first look\nat the benefits of using an API rate limiter:\n• Prevent resource starvation caused by Denial of Service (DoS) attack [1]. Almost all\nAPIs published by large tech companies enforce some form of rate limiting. For example,\nTwitter limits the number of

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

llm=init_chat_model("ollama:qwen3:8b")


## retrieve context and generate response

def rag_simple(query, retriever:RAGRetriever, llm, top_k=5):
    results=retriever.retrieve(query,top_k=top_k)

    # dedupe retrieved chunks by content (safety net against duplicate data)
    seen=set()
    unique=[]
    for doc in results:
        if doc['content'] not in seen:
            seen.add(doc['content'])
            unique.append(doc)

    context='\n\n'.join(doc['content'] for doc in unique) if unique else ""
    if not context:
        return "No relevant context found to answer the question"

    prompt=f"""Use the following context to answer the question concisely
            Context:
            {context}

            Question: {query}

            Answer:"""

    response=llm.invoke([prompt])
    return response.content

answer=rag_simple("what are different techniques of rate-limiting?", rag_retriever,llm)
print(answer)

Retrieving documents for query: what are different techniques of rate-limiting?
Top K: 5, Score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 185.93it/s]

generated embeddings with shape: (1, 384)
retrieved 5 documents (after filter)


The different techniques of rate-limiting include:  
1. **Token bucket**  
2. **Leaking bucket**  
3. **Fixed window counter**  
4. **Sliding window log**  
5. **Sliding window counter**  

These algorithms vary in their approach to tracking request rates and enforcing limits, with trade-offs in accuracy, latency, and memory usage.


In [9]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what are different techniques of rate-limiting?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: what are different techniques of rate-limiting?
Top K: 3, Score threshold: 0.1
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 127.95it/s]

generated embeddings with shape: (1, 384)
retrieved 3 documents (after filter)


Answer: The different techniques of rate-limiting include:  
1. **Token bucket**  
2. **Leaking bucket**  
3. **Fixed window counter**  
4. **Sliding window log**  
5. **Sliding window counter**  
Additionally, rate limiting can be applied at different layers (e.g., application vs. network) and categorized into **hard** (strict threshold) or **soft** (temporary exceedance allowed).
Sources: [{'source': 'System Design Interview by Alex Xu.pdf', 'page': 50, 'score': 0.39798295497894287, 'preview': 'CHAPTER 4: DESIGN A RATE LIMITER \nIn a network system, a rate limiter is used to control the rate of traffic sent by a client or a\nservice. In the HTTP world, a rate limiter limits the number of client requests allowed to be\nsent over a specified period. If the API request count exceeds the threshol...'}, {'source': 'System Design Interview by Alex Xu.pdf', 'page': 53, 'score': 0.39416396617889404, 'preview': 'Rate limiting can be implemented using different algorithms, and each of them has

In [10]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what are different techniques of rate-limiting?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: what are different techniques of rate-limiting?
Top K: 3, Score threshold: 0.1
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.62it/s]

generated embeddings with shape: (1, 384)
retrieved 3 documents (after filter)
Streaming answer:
Use the following context to answer the question concisely.
Context:
CHAPTER 4: DESIGN A RATE LIMITER 
In a network system, a rate limiter is used to control the rate of traffic sent by a client or a
service. In the HTTP world, a rate limiter limits the number of client requests allowed to be
sent over a specified peri

od. If the API request count exceeds the threshold defined by the
rate limiter, all the excess calls are blocked. Here are a few examples:
• A user can write no more than 2 posts per second.
• You can create a maximum of 10 accounts per day from the same IP address.
• You can claim rewards no more than 5 times per week from the same device.
In this chapter, you are asked to design a rate limiter. Before starting the design, we first look
at the benefits of using an API rate limiter:
• Prevent resource starvation caused by Denial of Service (DoS) attack [1]. Almost all
APIs published by large tech companies enforce some form of rate limiting. For example,
Twitter limits the number of tweets to 300 per 3 hours [2]. Google docs APIs have the

Rate limiting can be implemented using different algorithms, and each of them has distinct
pros and cons. Even though this chapter does not focus on algorithms, understanding them at
high-level helps to choose the right algorithm or combination of al